# 📊 Model Evaluation & Comparison

This notebook compares all three models side-by-side with comprehensive evaluations.

**Prerequisites**: Run all model notebooks first:
1. `preprocessing.ipynb`
2. `02_train_decision_tree.ipynb`
3. `03_train_xgboost.ipynb`
4. `04_train_hgnn.ipynb`

In [1]:
import sys, os
import warnings
warnings.filterwarnings('ignore')
import pickle

# Ensure PROJECT_P is on the path
PROJECT_ROOT = os.path.dirname(os.path.dirname(os.path.abspath('__file__')))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from IPython.display import Image, display

# Project modules
from src.config import *
from src.utils import get_logger, set_plot_style

logger = get_logger('Evaluation')
print('✅ Imports successful')

✅ Imports successful


## 1. Load All Model Results

In [2]:
# Load all model results
results = {}
models = ['decision_tree', 'xgboost', 'hgnn']
labels = ['Decision Tree', 'XGBoost', 'HGNN-ATT-TD']

for model_name, label in zip(models, labels):
    results_file = os.path.join(MODEL_DIR, f'{model_name}_results.pkl')
    
    if not os.path.exists(results_file):
        print(f'⚠️  Missing: {results_file}')
        continue
    
    with open(results_file, 'rb') as f:
        results[label] = pickle.load(f)
    
    print(f'✅ Loaded {label}')

print(f'\n📊 Loaded {len(results)} model results')

✅ Loaded Decision Tree
✅ Loaded XGBoost
✅ Loaded HGNN-ATT-TD

📊 Loaded 3 model results


## 2. Create Comparison Table

In [3]:
# Create comparison dataframe
comparison_data = []

for model_name, model_results in results.items():
    metrics = model_results['metrics']
    comparison_data.append({
        'Model': model_name,
        'Accuracy': f"{metrics['Accuracy']:.4f}",
        'Precision': f"{metrics['Precision']:.4f}",
        'Recall': f"{metrics['Recall']:.4f}",
        'F1-Score': f"{metrics['F1-Score']:.4f}",
        'ROC-AUC': f"{metrics['ROC-AUC']:.4f}",
    })

comp_df = pd.DataFrame(comparison_data)

print('\n' + '='*80)
print('           FINAL MODEL COMPARISON')
print('='*80)
print(comp_df.to_string(index=False))
print('='*80)


           FINAL MODEL COMPARISON
        Model Accuracy Precision Recall F1-Score ROC-AUC
Decision Tree   0.9366    0.2929 0.5457   0.3812  0.8390
      XGBoost   0.9326    0.3111 0.7266   0.4357  0.9198
  HGNN-ATT-TD   0.9642    0.4962 0.2057   0.2908  0.8186


## 3. Generate Comparison Plots

In [6]:
import os
import pickle
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import precision_recall_curve, average_precision_score

# styling helper from project (if available)
try:
    from src.utils import set_plot_style
    set_plot_style()
except Exception:
    plt.style.use("seaborn-whitegrid")

# paths
preprocess_file = os.path.join(MODEL_DIR, "preprocessed_data.pkl")
if not os.path.exists(preprocess_file):
    raise FileNotFoundError(f"Missing {preprocess_file} - run preprocessing first")

with open(preprocess_file, "rb") as f:
    prep = pickle.load(f)
y_test = prep["y_test"]

models = [
    ("decision_tree", "Decision Tree", "#1f77b4"),
    ("xgboost", "XGBoost", "#ff7f0e"),
    ("hgnn", "HGNN-ATT-TD", "#2ca02c"),
]

os.makedirs(EVAL_DIR, exist_ok=True)

for short_name, label, color in models:
    results_path = os.path.join(MODEL_DIR, f"{short_name}_results.pkl")
    out_path = os.path.join(EVAL_DIR, f"pr_curve_{short_name}.png")

    if not os.path.exists(results_path):
        print(f"⚠️  Missing results file: {results_path} — skipping {label}")
        continue

    with open(results_path, "rb") as f:
        res = pickle.load(f)

    y_pred_proba = res.get("y_pred_proba")
    if y_pred_proba is None:
        print(f"⚠️  {label} has no 'y_pred_proba' field — skipping")
        continue

    # handle length mismatch: prefer exact match; if prediction array is longer, use tail
    if len(y_pred_proba) != len(y_test):
        if len(y_pred_proba) > len(y_test):
            print(
                f"⚠️  Length mismatch for {label}: "
                f"y_pred_proba={len(y_pred_proba)}, y_test={len(y_test)}. "
                "Using last len(y_test) predictions as fallback."
            )
            y_pred_proba = np.asarray(y_pred_proba)[-len(y_test) :]
        else:
            print(
                f"⚠️  {label} predictions shorter ({len(y_pred_proba)}) than y_test ({len(y_test)}) — skipping"
            )
            continue

    precision, recall, _ = precision_recall_curve(y_test, y_pred_proba)
    ap = average_precision_score(y_test, y_pred_proba)

    plt.figure(figsize=(8, 6))
    plt.plot(recall, precision, color=color, lw=2, label=f"{label} (AP={ap:.3f})")
    no_skill = float(np.mean(y_test))
    plt.hlines(no_skill, 0, 1, linestyles="--", colors="gray", label=f"No Skill (pos rate={no_skill:.3f})")
    plt.xlabel("Recall", fontsize=12, fontweight="bold")
    plt.ylabel("Precision", fontsize=12, fontweight="bold")
    plt.title(f"Precision-Recall Curve — {label}", fontsize=14, fontweight="bold")
    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])
    plt.grid(alpha=0.3)
    plt.legend(loc="best", fontsize=11)
    plt.tight_layout()
    plt.savefig(out_path, dpi=150, bbox_inches="tight")
    plt.close()
    print(f"✅ Saved {label} PR curve to {out_path}")

✅ Saved Decision Tree PR curve to c:\Users\Lekshmi Priya\OneDrive\Documents\GitHub\Credit-Card-Fraud-Detection-System\outputs\pr_curve_decision_tree.png
✅ Saved XGBoost PR curve to c:\Users\Lekshmi Priya\OneDrive\Documents\GitHub\Credit-Card-Fraud-Detection-System\outputs\pr_curve_xgboost.png
⚠️  HGNN-ATT-TD predictions shorter (8859) than y_test (26575) — skipping


In [8]:
import os, pickle, numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import precision_recall_curve, average_precision_score

try:
    from src.utils import set_plot_style
    set_plot_style()
except Exception:
    plt.style.use("seaborn-whitegrid")

preprocess_file = os.path.join(MODEL_DIR, "preprocessed_data.pkl")
with open(preprocess_file, "rb") as f:
    prep = pickle.load(f)
y_test = np.asarray(prep["y_test"])

hgnn_path = os.path.join(MODEL_DIR, "hgnn_results.pkl")
with open(hgnn_path, "rb") as f:
    res = pickle.load(f)

y_pred_proba = np.asarray(res.get("y_pred_proba"))
if y_pred_proba is None:
    raise RuntimeError("hgnn_results.pkl missing 'y_pred_proba'")

# Align lengths by taking the last min_len samples from each
min_len = min(len(y_test), len(y_pred_proba))
if len(y_test) != len(y_pred_proba):
    print(f"⚠️ Length mismatch: y_test={len(y_test)}, y_pred_proba={len(y_pred_proba)}. Aligning last {min_len} samples.")
y_test_aligned = y_test[-min_len:]
y_pred_proba_aligned = y_pred_proba[-min_len:]

precision, recall, _ = precision_recall_curve(y_test_aligned, y_pred_proba_aligned)
ap = average_precision_score(y_test_aligned, y_pred_proba_aligned)

os.makedirs(EVAL_DIR, exist_ok=True)
out_path = os.path.join(EVAL_DIR, "pr_curve_hgnn_fixed.png")

plt.figure(figsize=(8, 6))
plt.plot(recall, precision, color="#2ca02c", lw=2, label=f"HGNN-ATT-TD (AP={ap:.3f})")
no_skill = float(np.mean(y_test_aligned))
plt.hlines(no_skill, 0, 1, linestyles="--", colors="gray", label=f"No Skill (pos rate={no_skill:.3f})")
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Precision-Recall Curve — HGNN-ATT-TD")
plt.xlim([0, 1])
plt.ylim([0, 1.05])
plt.grid(alpha=0.3)
plt.legend(loc="best")
plt.tight_layout()
plt.savefig(out_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"✅ Saved HGNN PR curve to {out_path}")

⚠️ Length mismatch: y_test=26575, y_pred_proba=8859. Aligning last 8859 samples.
✅ Saved HGNN PR curve to c:\Users\Lekshmi Priya\OneDrive\Documents\GitHub\Credit-Card-Fraud-Detection-System\outputs\pr_curve_hgnn_fixed.png


In [12]:
from sklearn.metrics import confusion_matrix

# Confusion Matrices
fig, axes = plt.subplots(1, 3, figsize=(18, 4))

for idx, (model_name, model_results) in enumerate(results.items()):
    y_pred = model_results['y_pred']
    y_true = y_test[-len(y_pred):] if len(y_pred) <= len(y_test) else y_test
    y_pred = y_pred[-len(y_true):]
    cm = confusion_matrix(y_true, y_pred)
    
    # Plot
    im = axes[idx].imshow(cm, cmap='Blues', aspect='auto')
    axes[idx].set_title(f'{model_name}\nConfusion Matrix', fontweight='bold')
    axes[idx].set_ylabel('True Label')
    axes[idx].set_xlabel('Predicted Label')
    axes[idx].set_xticks([0, 1])
    axes[idx].set_yticks([0, 1])
    axes[idx].set_xticklabels(['Legit', 'Fraud'])
    axes[idx].set_yticklabels(['Legit', 'Fraud'])
    
    # Add text
    for i in range(2):
        for j in range(2):
            text = axes[idx].text(j, i, cm[i, j], ha='center', va='center',
                                color='white' if cm[i, j] > cm.max()/2 else 'black',
                                fontsize=12, fontweight='bold')
    
    plt.colorbar(im, ax=axes[idx])

plt.tight_layout()
plt.savefig(os.path.join(EVAL_DIR, '03_confusion_matrices.png'), dpi=150, bbox_inches='tight')
plt.show()
print('✅ Confusion matrices saved')

✅ Confusion matrices saved
